# Working with JSON

JSON is the format AI APIs use to send and receive data.

**You will learn to:**

- Convert between JSON text and Python objects
- Read and write JSON files
- Navigate nested data safely
- Handle invalid JSON
- Extract JSON from a model's text reply

## 1. What is JSON?

JSON (JavaScript Object Notation) is plain text that looks like Python dictionaries and lists.

| JSON | Python |
|---|---|
| object `{}` | `dict` |
| array `[]` | `list` |
| string | `str` |
| number | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

JSON is stricter than Python: keys must be strings in **double quotes**, and there are no trailing commas or comments.

## 2. Text to Python and back

In [ ]:
import json

text = '{"name": "Ada", "age": 36, "skills": ["math", "code"], "active": true, "manager": null}'

data = json.loads(text)                 # loads = load from string
print(data)
print(type(data), data["skills"][0], data["active"], data["manager"])

In [ ]:
record = {"model": "sonnet", "tags": ["ai", "demo"], "stream": False, "note": None, "price": 3.0}

as_text = json.dumps(record)            # dumps = dump to string
print(as_text)

print(json.dumps(record, indent=2))                   # pretty
print(json.dumps(record, sort_keys=True))             # stable key order
print(json.dumps({"city": "Zürich"}, ensure_ascii=False))

Not everything converts. Use `default=str` as a quick fallback for dates and similar values.

In [ ]:
from datetime import date

payload = {"created": date(2026, 10, 26)}

try:
    json.dumps(payload)
except TypeError as error:
    print("Error:", error)

print(json.dumps(payload, default=str))

## 3. JSON files

In [ ]:
settings = {"model": "sonnet", "max_tokens": 1024, "tags": ["day-2"]}

with open("settings.json", "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2)             # dump / load work with files

with open("settings.json", encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded == settings)
print(open("settings.json", encoding="utf-8").read())

In [ ]:
from pathlib import Path

Path("settings.json").unlink()      # clean up

## 4. Navigate nested data

The next structure mirrors the shape of a typical model API response.

In [ ]:
response_text = """
{
  "id": "msg_123",
  "model": "example-model",
  "content": [
    {"type": "text", "text": "Hello! How can I help?"}
  ],
  "usage": {"input_tokens": 12, "output_tokens": 9},
  "stop_reason": "end_turn"
}
"""

response = json.loads(response_text)

print(response["content"][0]["text"])
print(response["usage"]["input_tokens"] + response["usage"]["output_tokens"])

Use `.get()` when a key may be missing, so one gap does not crash the program.

In [ ]:
print(response.get("stop_sequence"))                         # None
print(response.get("usage", {}).get("cache_tokens", 0))      # chained safe lookup

try:
    response["stop_sequence"]
except KeyError as error:
    print("KeyError:", error)

## 5. Process a list of records

In [ ]:
orders_text = """
[
  {"id": 1, "customer": "Ada",   "total": 120.5, "status": "paid"},
  {"id": 2, "customer": "Linus", "total": 80.0,  "status": "refunded"},
  {"id": 3, "customer": "Grace", "total": 200.0, "status": "paid"}
]
"""

orders = json.loads(orders_text)

paid = [o for o in orders if o["status"] == "paid"]
revenue = sum(o["total"] for o in paid)

print(f"{len(paid)} paid orders, revenue {revenue:,.2f}")
print([o["customer"] for o in paid])

summary = {"paid_count": len(paid), "revenue": revenue}
print(json.dumps(summary))

## 6. Invalid JSON

In [ ]:
bad_inputs = [
    "{'name': 'Ada'}",          # single quotes
    '{"name": "Ada",}',         # trailing comma
    "",                         # empty
]

for bad in bad_inputs:
    try:
        json.loads(bad)
    except json.JSONDecodeError as error:
        print(f"Invalid JSON ({error.msg}) at position {error.pos}")

### Validate the shape

Valid JSON can still have the wrong **shape**. Check required keys and types before you trust it.

In [ ]:
def parse_user(text):
    """Return a user dict, or raise ValueError with a clear reason."""
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError(f"not valid JSON: {error.msg}") from error

    if not isinstance(data, dict):
        raise ValueError("expected a JSON object")
    for key, expected_type in {"name": str, "age": int}.items():
        if key not in data:
            raise ValueError(f"missing key {key!r}")
        if not isinstance(data[key], expected_type):
            raise ValueError(f"{key!r} must be {expected_type.__name__}")
    return data


for sample in ['{"name": "Ada", "age": 36}', '{"name": "Ada"}', '{"name": "Ada", "age": "36"}', "oops", "[1, 2]"]:
    try:
        print("OK  ", parse_user(sample))
    except ValueError as error:
        print("FAIL", error)

## 7. Extract JSON from a model reply

Even when you ask for JSON, a model may wrap it in a code fence or add a sentence around it.
This helper pulls out the first JSON object it can find.

In [ ]:
import re


def extract_json(reply):
    """Return the first JSON object found in `reply`, parsed into a dict."""
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", reply, re.DOTALL)
    candidate = fenced.group(1) if fenced else None
    if candidate is None:
        start, end = reply.find("{"), reply.rfind("}")
        if start == -1 or end == -1:
            raise ValueError("no JSON object found")
        candidate = reply[start:end + 1]
    return json.loads(candidate)


fence = "`" * 3
replies = [
    'Here you go: {"sentiment": "positive", "score": 0.92} Hope that helps!',
    f'{fence}json\n{{"sentiment": "negative", "score": 0.1}}\n{fence}',
]

for reply in replies:
    print(extract_json(reply))

> **Later in the course:** structured outputs and schema validation give stronger guarantees than parsing text. The habits here still matter: always handle bad JSON.

## Try it

1. Convert `{"name": "Ada", "scores": [90, 85]}` to indented JSON text and back, and confirm the two dictionaries are equal.
2. Write the orders from section 5 to `orders.json`, read the file back and print the total of all orders.
3. Extend `parse_user` to require `email` containing `@`.

In [ ]:
# Your answers here